In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("HospitalData").getOrCreate()

In [ ]:
%%writefile patients.csv
patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active

Overwriting patients.csv


In [ ]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,fee,consult
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2500,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending

Overwriting appointments.csv


In [ ]:
%%writefile patient_preferences.json
[
{
"patient_id":101,
"preferred_hospital":"Apollo",
"contact":{
"phone":"9876500011",
"email":"rahul@gmail.com"
}
},
{
"patient_id":102,
"preferred_hospital":"Yashoda",
"contact":{
"phone":null,
"email":"priya@gmail.com"
}
},
{
"patient_id":103,
"preferred_hospital":"Care",
"contact":{
"phone":"9876500013",
"email":null
}
},
{
"patient_id":104,
"preferred_hospital":null,
"contact":{
"phone":"9876500014",
"email":"sneha@gmail.com"
}
}
]

Overwriting patient_preferences.json


CSV ingestion

In [ ]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

In [ ]:
appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [ ]:
patients_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)



In [ ]:
appointments_df.printSchema()

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: timestamp (nullable = true)
 |-- fee: integer (nullable = true)
 |-- consult: string (nullable = true)



In [ ]:
print(patients_df.count())
print(appointments_df.count())

10
10


In [ ]:
patients_df.show(5, truncate=False)
appointments_df.show(5, truncate=False)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows
+--------------+----------+-----------+-----------+-------------------+-------+
|appointment_id|patient_id|doctor_name|department |appointment_date   |consult|
+--------------+----------+-----------+-----------+-------------------+-------+
|5001          |101       |Dr. Ramesh |Cardiology |2

In [ ]:
patients_df.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [ ]:
appointments_df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [ ]:
patients_df.write.mode("overwrite").parquet("patients_parquet")

In [ ]:
patients_parquet_df = spark.read.parquet("patients_parquet")
patients_parquet_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [ ]:
csv_count = patients_df.count()
parquet_count = patients_parquet_df.count()
print(csv_count)
print(parquet_count)
if csv_count == parquet_count:
    print("Counts Match")
else:
    print("Counts Do Not Match")

10
10
Counts Match


filtering

In [ ]:
from pyspark.sql.functions import col
patients_df.filter(col("city") == "Hyderabad").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
patients_df.filter(col("gender") == "Female").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
patients_df.filter(col("age") > 40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
appointments_df.filter(col("consult") == "Completed").show()

+--------------+----------+-----------+-----------+-------------------+----+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|
+--------------+----------+-----------+-----------+-------------------+----+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|2025-02-05 00:00:00|1500|Completed|
+--------------+----------+-----------+-----------+-------------------+----+---------+



In [ ]:
appointments_df.filter(col("consult") == "Pending").show()

+--------------+----------+-----------+-----------+-------------------+----+-------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|consult|
+--------------+----------+-----------+-----------+-------------------+----+-------+
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|2025-02-10 00:00:00|NULL|Pending|
+--------------+----------+-----------+-----------+-------------------+----+-------+



In [ ]:
appointments_df.filter(col("fee") > 1500).show()

+--------------+----------+-----------+-----------+-------------------+----+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|
+--------------+----------+-----------+-----------+-------------------+----+---------+
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|2500|Completed|
+--------------+----------+-----------+-----------+-------------------+----+---------+



In [ ]:
patients_df.filter(col("insurance_status") == "Active").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
patients_df.filter(col("insurance_status") == "Inactive").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
patients_df.filter(col("blood_group") == "O+").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
appointments_df.filter(col("department") == "Cardiology").show()

+--------------+----------+-----------+----------+-------------------+----+---------+
|appointment_id|patient_id|doctor_name|department|   appointment_date| fee|  consult|
+--------------+----------+-----------+----------+-------------------+----+---------+
|          5001|       101| Dr. Ramesh|Cardiology|2025-01-10 00:00:00|1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|2025-01-20 00:00:00|1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|2025-02-05 00:00:00|1500|Completed|
+--------------+----------+-----------+----------+-------------------+----+---------+



null handling

In [ ]:
patients_df.filter(col("city").isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [ ]:
patients_df.filter(col("blood_group").isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [ ]:
appointments_df.filter(col("fee").isNull()).show()

+--------------+----------+-----------+-----------+-------------------+----+-------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|consult|
+--------------+----------+-----------+-----------+-------------------+----+-------+
|          5010|       108|  Dr. Anita|Dermatology|2025-02-10 00:00:00|NULL|Pending|
+--------------+----------+-----------+-----------+-------------------+----+-------+



In [ ]:
from pyspark.sql.functions import col, sum, when
patients_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in patients_df.columns
]).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+



In [ ]:
patients_df = patients_df.fillna({
    "city": "Unknown"
})
patients_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [ ]:
patients_df = patients_df.fillna({
    "blood_group": "Not Available"
})
patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [ ]:
appointments_df = appointments_df.fillna({
    "fee": 0
})
appointments_df.show()

+--------------+----------+-----------+-----------+-------------------+----+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|
+--------------+----------+-----------+-----------+-------------------+----+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|2500|Completed|
|          5009|       120| Dr. Ramesh| Car

In [ ]:
appointments_no_null_fee = appointments_df.na.drop(
    subset=["fee"]
)
appointments_no_null_fee.show()

+--------------+----------+-----------+-----------+-------------------+----+---------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|
+--------------+----------+-----------+-----------+-------------------+----+---------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|2500|Completed|
|          5009|       120| Dr. Ramesh| Car

In [ ]:
from pyspark.sql.functions import when
patients_quality_df = patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull() |
        col("blood_group").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)
patients_quality_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|           Complete|
|       108|  Meera 

In [ ]:
patients_quality_df.groupBy(
    "data_quality_status"
).count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   10|
+-------------------+-----+



Built in functions

In [ ]:
from pyspark.sql.functions import *
patients_df.select(
    "patient_name",
    upper("patient_name").alias("name_upper")
).show()

+------------+------------+
|patient_name|  name_upper|
+------------+------------+
|Rahul Sharma|RAHUL SHARMA|
| Priya Reddy| PRIYA REDDY|
|  Amit Kumar|  AMIT KUMAR|
| Sneha Patel| SNEHA PATEL|
|  Farhan Ali|  FARHAN ALI|
|  Neha Singh|  NEHA SINGH|
| Arjun Verma| ARJUN VERMA|
|  Meera Nair|  MEERA NAIR|
|   Kiran Rao|   KIRAN RAO|
| Nisha Reddy| NISHA REDDY|
+------------+------------+



In [ ]:
patients_df.select(
    "patient_name",
    lower("patient_name").alias("name_lower")
).show()

+------------+------------+
|patient_name|  name_lower|
+------------+------------+
|Rahul Sharma|rahul sharma|
| Priya Reddy| priya reddy|
|  Amit Kumar|  amit kumar|
| Sneha Patel| sneha patel|
|  Farhan Ali|  farhan ali|
|  Neha Singh|  neha singh|
| Arjun Verma| arjun verma|
|  Meera Nair|  meera nair|
|   Kiran Rao|   kiran rao|
| Nisha Reddy| nisha reddy|
+------------+------------+



In [ ]:
patients_df.select(
    "patient_name",
    length("patient_name").alias("name_length")
).show()

+------------+-----------+
|patient_name|name_length|
+------------+-----------+
|Rahul Sharma|         12|
| Priya Reddy|         11|
|  Amit Kumar|         10|
| Sneha Patel|         11|
|  Farhan Ali|         10|
|  Neha Singh|         10|
| Arjun Verma|         11|
|  Meera Nair|         10|
|   Kiran Rao|          9|
| Nisha Reddy|         11|
+------------+-----------+



In [ ]:
patients_df.select(
    "patient_name",
    substring("patient_name", 1, 3).alias("first_3_letters")
).show()

+------------+---------------+
|patient_name|first_3_letters|
+------------+---------------+
|Rahul Sharma|            Rah|
| Priya Reddy|            Pri|
|  Amit Kumar|            Ami|
| Sneha Patel|            Sne|
|  Farhan Ali|            Far|
|  Neha Singh|            Neh|
| Arjun Verma|            Arj|
|  Meera Nair|            Mee|
|   Kiran Rao|            Kir|
| Nisha Reddy|            Nis|
+------------+---------------+



In [ ]:
patients_age_df = patients_df.withColumn(
    "age_group",
    when(col("age") < 18, "Child")
    .when(col("age") < 40, "Adult")
    .otherwise("Senior")
)
patients_age_df.show()

+----------+------------+---------+---+------+-------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|age_group|
+----------+------------+---------+---+------+-------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|    Adult|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|    Adult|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|   Senior|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|    Adult|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|   Senior|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|    Adult|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|    Adult|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|   Senior|
|       109|   Kiran Rao|Hyderab

In [ ]:
patients_ins_df = patients_df.withColumn(
    "insurance_flag",
    when(col("insurance_status") == "Active", 1)
    .otherwise(0)
)
patients_ins_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|insurance_flag|
+----------+------------+---------+---+------+-------------+----------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|             1|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|             0|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|             1|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|             0|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|             1|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          

In [ ]:
patients_senior_df = patients_df.withColumn(
    "senior_citizen",
    when(col("age") >= 60, "Yes")
    .otherwise("No")
)
patients_senior_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|            No|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|            No|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|            No|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|            No|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          

In [ ]:
patients_df.select(
    concat_ws(" - ",
              col("patient_name"),
              col("city")).alias("patient_city")
).show(truncate=False)

+------------------------+
|patient_city            |
+------------------------+
|Rahul Sharma - Hyderabad|
|Priya Reddy - Bangalore |
|Amit Kumar - Mumbai     |
|Sneha Patel - Chennai   |
|Farhan Ali - Delhi      |
|Neha Singh - Unknown    |
|Arjun Verma - Pune      |
|Meera Nair - Kochi      |
|Kiran Rao - Hyderabad   |
|Nisha Reddy - Bangalore |
+------------------------+



In [ ]:
patients_df.select(
    "patient_name",
    trim("patient_name").alias("trimmed_name")
).show()

+------------+------------+
|patient_name|trimmed_name|
+------------+------------+
|Rahul Sharma|Rahul Sharma|
| Priya Reddy| Priya Reddy|
|  Amit Kumar|  Amit Kumar|
| Sneha Patel| Sneha Patel|
|  Farhan Ali|  Farhan Ali|
|  Neha Singh|  Neha Singh|
| Arjun Verma| Arjun Verma|
|  Meera Nair|  Meera Nair|
|   Kiran Rao|   Kiran Rao|
| Nisha Reddy| Nisha Reddy|
+------------+------------+



In [ ]:
patients_df.select(
    "city",
    upper("city").alias("city_upper")
).show()

+---------+----------+
|     city|city_upper|
+---------+----------+
|Hyderabad| HYDERABAD|
|Bangalore| BANGALORE|
|   Mumbai|    MUMBAI|
|  Chennai|   CHENNAI|
|    Delhi|     DELHI|
|  Unknown|   UNKNOWN|
|     Pune|      PUNE|
|    Kochi|     KOCHI|
|Hyderabad| HYDERABAD|
|Bangalore| BANGALORE|
+---------+----------+



Groupby and Aggregations

In [ ]:
patients_df.groupBy("city") \
    .count() \
    .show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    1|
|  Unknown|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [ ]:
patients_df.groupBy("gender") \
    .count() \
    .show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [ ]:
patients_df.groupBy("blood_group") \
    .count() \
    .show()

+-------------+-----+
|  blood_group|count|
+-------------+-----+
|          AB+|    1|
|           O+|    2|
|           O-|    1|
|           B+|    2|
|           A+|    3|
|Not Available|    1|
+-------------+-----+



In [ ]:
appointments_df.groupBy("department") \
    .count() \
    .show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [ ]:
patients_df.groupBy("city") \
    .agg(
        avg("age").alias("avg_age")
    ) \
    .show()

+---------+-------+
|     city|avg_age|
+---------+-------+
|Bangalore|   35.0|
|    Kochi|   48.0|
|  Chennai|   31.0|
|   Mumbai|   42.0|
|  Unknown|   38.0|
|     Pune|   26.0|
|    Delhi|   55.0|
|Hyderabad|   34.0|
+---------+-------+



In [ ]:
patients_df.groupBy("city") \
    .agg(
        max("age").alias("max_age")
    ) \
    .show()

+---------+-------+
|     city|max_age|
+---------+-------+
|Bangalore|     41|
|    Kochi|     48|
|  Chennai|     31|
|   Mumbai|     42|
|  Unknown|     38|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     35|
+---------+-------+



In [ ]:
patients_df.groupBy("city") \
    .agg(
        min("age").alias("min_age")
    ) \
    .show()

+---------+-------+
|     city|min_age|
+---------+-------+
|Bangalore|     29|
|    Kochi|     48|
|  Chennai|     31|
|   Mumbai|     42|
|  Unknown|     38|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     33|
+---------+-------+



In [ ]:
appointments_df.groupBy("department") \
    .agg(
        round(avg("fee"), 2).alias("avg_fee")
    ) \
    .show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 666.67|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [ ]:
appointments_df.groupBy("department") \
    .agg(
        sum("fee").alias("total_fee")
    ) \
    .show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [ ]:
appointments_df.groupBy("department") \
    .agg(
        sum("fee").alias("total_revenue")
    ) \
    .orderBy(desc("total_revenue")) \
    .show(1)

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
+-----------+-------------+
only showing top 1 row


Joins

In [ ]:
inner_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="inner"
)
inner_df.show()

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date| fee|  consult|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|          5004| Dr. Rames

In [ ]:
left_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="left"
)
left_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date| fee|  consult|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|          5002| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|          5

In [ ]:
right_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="right"
)
right_df.show()

+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date| fee|  consult|
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|       103|  Amit Kumar|   Mumbai|  42|  Male|         B+|        Inactive|          5004| Dr

In [ ]:
full_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="full"
)
full_df.show()

+----------+------------+---------+----+------+-------------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|patient_id|patient_name|     city| age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date| fee|  consult|
+----------+------------+---------+----+------+-------------+----------------+--------------+-----------+-----------+-------------------+----+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|           A+|          Active|          5002| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|
|       103|  Amit Kumar|   Mumbai|  42|  Male|           B+|        Inactive|    

In [ ]:
patients_no_appt = patients_df.join(
    appointments_df,
    on="patient_id",
    how="left"
).filter(
    col("appointment_id").isNull()
)
patients_no_appt.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+----------+----------------+----+-------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name|department|appointment_date| fee|consult|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+----------+----------------+----+-------+
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|          NULL|       NULL|      NULL|            NULL|NULL|   NULL|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|          NULL|       NULL|      NULL|            NULL|NULL|   NULL|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+----------+----------------+----+-------+



In [ ]:
orphan_appointments = appointments_df.join(
    patients_df,
    on="patient_id",
    how="left"
).filter(
    col("patient_name").isNull()
)
orphan_appointments.show()

+----------+--------------+-----------+----------+-------------------+----+---------+------------+----+----+------+-----------+----------------+
|patient_id|appointment_id|doctor_name|department|   appointment_date| fee|  consult|patient_name|city| age|gender|blood_group|insurance_status|
+----------+--------------+-----------+----------+-------------------+----+---------+------------+----+----+------+-----------+----------------+
|       120|          5009| Dr. Ramesh|Cardiology|2025-02-05 00:00:00|1500|Completed|        NULL|NULL|NULL|  NULL|       NULL|            NULL|
+----------+--------------+-----------+----------+-------------------+----+---------+------------+----+----+------+-----------+----------------+



In [ ]:
appointments_df.groupBy("patient_id") \
    .count() \
    .withColumnRenamed("count", "appointment_count") \
    .show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [ ]:
patient_fee_df = appointments_df.groupBy("patient_id") \
    .agg(
        sum("fee").alias("total_fee_paid")
    )
patient_fee_df.show()

+----------+--------------+
|patient_id|total_fee_paid|
+----------+--------------+
|       108|             0|
|       101|          2500|
|       103|          1500|
|       120|          1500|
|       107|          2000|
|       102|          2000|
|       105|          1000|
|       110|          2500|
|       104|          2500|
+----------+--------------+



In [ ]:
highest_spender = patients_df.join(
    appointments_df,
    on="patient_id"
).groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("fee").alias("total_spent")
).orderBy(
    desc("total_spent")
)
highest_spender.show(1)

+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       101|Rahul Sharma|       2500|
+----------+------------+-----------+
only showing top 1 row


In [ ]:
patient_appointment_count = patients_df.join(
    appointments_df,
    on="patient_id",
    how="left"
).groupBy(
    "patient_id",
    "patient_name"
).agg(
    count("appointment_id").alias("appointment_count")
)
patient_appointment_count.show()

+----------+------------+-----------------+
|patient_id|patient_name|appointment_count|
+----------+------------+-----------------+
|       107| Arjun Verma|                1|
|       108|  Meera Nair|                1|
|       109|   Kiran Rao|                0|
|       110| Nisha Reddy|                1|
|       105|  Farhan Ali|                1|
|       101|Rahul Sharma|                2|
|       104| Sneha Patel|                1|
|       103|  Amit Kumar|                1|
|       106|  Neha Singh|                0|
|       102| Priya Reddy|                1|
+----------+------------+-----------------+



Window functions

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
patient_spending = patients_df.join(
    appointments_df,
    on="patient_id"
).groupBy(
    "patient_id",
    "patient_name",
    "city"
).agg(
    sum("fee").alias("total_fee")
)
patient_spending.show()

+----------+------------+---------+---------+
|patient_id|patient_name|     city|total_fee|
+----------+------------+---------+---------+
|       108|  Meera Nair|    Kochi|        0|
|       110| Nisha Reddy|Bangalore|     2500|
|       101|Rahul Sharma|Hyderabad|     2500|
|       102| Priya Reddy|Bangalore|     2000|
|       103|  Amit Kumar|   Mumbai|     1500|
|       105|  Farhan Ali|    Delhi|     1000|
|       104| Sneha Patel|  Chennai|     2500|
|       107| Arjun Verma|     Pune|     2000|
+----------+------------+---------+---------+



In [ ]:
rank_window = Window.orderBy(desc("total_fee"))
patient_spending.withColumn(
    "rank",
    rank().over(rank_window)
).show()

+----------+------------+---------+---------+----+
|patient_id|patient_name|     city|total_fee|rank|
+----------+------------+---------+---------+----+
|       110| Nisha Reddy|Bangalore|     2500|   1|
|       101|Rahul Sharma|Hyderabad|     2500|   1|
|       104| Sneha Patel|  Chennai|     2500|   1|
|       102| Priya Reddy|Bangalore|     2000|   4|
|       107| Arjun Verma|     Pune|     2000|   4|
|       103|  Amit Kumar|   Mumbai|     1500|   6|
|       105|  Farhan Ali|    Delhi|     1000|   7|
|       108|  Meera Nair|    Kochi|        0|   8|
+----------+------------+---------+---------+----+



In [ ]:
patient_spending.withColumn(
    "dense_rank",
    dense_rank().over(rank_window)
).show()

+----------+------------+---------+---------+----------+
|patient_id|patient_name|     city|total_fee|dense_rank|
+----------+------------+---------+---------+----------+
|       110| Nisha Reddy|Bangalore|     2500|         1|
|       101|Rahul Sharma|Hyderabad|     2500|         1|
|       104| Sneha Patel|  Chennai|     2500|         1|
|       102| Priya Reddy|Bangalore|     2000|         2|
|       107| Arjun Verma|     Pune|     2000|         2|
|       103|  Amit Kumar|   Mumbai|     1500|         3|
|       105|  Farhan Ali|    Delhi|     1000|         4|
|       108|  Meera Nair|    Kochi|        0|         5|
+----------+------------+---------+---------+----------+



In [ ]:
patient_spending.withColumn(
    "row_num",
    row_number().over(rank_window)
).show()

+----------+------------+---------+---------+-------+
|patient_id|patient_name|     city|total_fee|row_num|
+----------+------------+---------+---------+-------+
|       110| Nisha Reddy|Bangalore|     2500|      1|
|       101|Rahul Sharma|Hyderabad|     2500|      2|
|       104| Sneha Patel|  Chennai|     2500|      3|
|       102| Priya Reddy|Bangalore|     2000|      4|
|       107| Arjun Verma|     Pune|     2000|      5|
|       103|  Amit Kumar|   Mumbai|     1500|      6|
|       105|  Farhan Ali|    Delhi|     1000|      7|
|       108|  Meera Nair|    Kochi|        0|      8|
+----------+------------+---------+---------+-------+



In [ ]:
patient_spending.orderBy(
    desc("total_fee")
).show(1)

+----------+------------+---------+---------+
|patient_id|patient_name|     city|total_fee|
+----------+------------+---------+---------+
|       101|Rahul Sharma|Hyderabad|     2500|
+----------+------------+---------+---------+
only showing top 1 row


In [ ]:
patient_spending.orderBy(
    desc("total_fee")
).show(3)

+----------+------------+---------+---------+
|patient_id|patient_name|     city|total_fee|
+----------+------------+---------+---------+
|       110| Nisha Reddy|Bangalore|     2500|
|       101|Rahul Sharma|Hyderabad|     2500|
|       104| Sneha Patel|  Chennai|     2500|
+----------+------------+---------+---------+
only showing top 3 rows


In [ ]:
city_window = Window.partitionBy(
    "city"
).orderBy(
    desc("total_fee")
)
highest_city = patient_spending.withColumn(
    "rank",
    rank().over(city_window)
).filter(
    col("rank") == 1
)
highest_city.show()

+----------+------------+---------+---------+----+
|patient_id|patient_name|     city|total_fee|rank|
+----------+------------+---------+---------+----+
|       110| Nisha Reddy|Bangalore|     2500|   1|
|       104| Sneha Patel|  Chennai|     2500|   1|
|       105|  Farhan Ali|    Delhi|     1000|   1|
|       101|Rahul Sharma|Hyderabad|     2500|   1|
|       108|  Meera Nair|    Kochi|        0|   1|
|       103|  Amit Kumar|   Mumbai|     1500|   1|
|       107| Arjun Verma|     Pune|     2000|   1|
+----------+------------+---------+---------+----+



In [ ]:
city_window_low = Window.partitionBy(
    "city"
).orderBy(
    asc("total_fee")
)
lowest_city = patient_spending.withColumn(
    "rank",
    rank().over(city_window_low)
).filter(
    col("rank") == 1
)
lowest_city.show()

+----------+------------+---------+---------+----+
|patient_id|patient_name|     city|total_fee|rank|
+----------+------------+---------+---------+----+
|       102| Priya Reddy|Bangalore|     2000|   1|
|       104| Sneha Patel|  Chennai|     2500|   1|
|       105|  Farhan Ali|    Delhi|     1000|   1|
|       101|Rahul Sharma|Hyderabad|     2500|   1|
|       108|  Meera Nair|    Kochi|        0|   1|
|       103|  Amit Kumar|   Mumbai|     1500|   1|
|       107| Arjun Verma|     Pune|     2000|   1|
+----------+------------+---------+---------+----+



In [ ]:
running_window = Window.orderBy(
    "appointment_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)
appointments_df.withColumn(
    "running_total",
    sum("fee").over(running_window)
).show()

+--------------+----------+-----------+-----------+-------------------+----+---------+-------------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|running_total|
+--------------+----------+-----------+-----------+-------------------+----+---------+-------------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|         1500|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|         3500|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|         4500|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|1500|Cancelled|         6000|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|         8500|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|1000|  Pending|         9500|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|     

In [ ]:
appointments_df.withColumn(
    "next_fee",
    lead("fee", 1).over(
        Window.orderBy("appointment_id")
    )
).show()

+--------------+----------+-----------+-----------+-------------------+----+---------+--------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|next_fee|
+--------------+----------+-----------+-----------+-------------------+----+---------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|    2000|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|    1000|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|    1500|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|1500|Cancelled|    2500|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|    1000|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|1000|  Pending|    2000|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|    2500|
|          5008|       110|  Dr. Priya|O

In [ ]:
appointments_df.withColumn(
    "previous_fee",
    lag("fee", 1).over(
        Window.orderBy("appointment_id")
    )
).show()


+--------------+----------+-----------+-----------+-------------------+----+---------+------------+
|appointment_id|patient_id|doctor_name| department|   appointment_date| fee|  consult|previous_fee|
+--------------+----------+-----------+-----------+-------------------+----+---------+------------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|        NULL|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|2000|Completed|        1500|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|        2000|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|1500|Cancelled|        1000|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|2500|Completed|        1500|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|1000|  Pending|        2500|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|2000|Completed|        1000|


JSON Processing

In [ ]:
preferences_df = spark.read.option(
    "multiline",
    "true"
).json("patient_preferences.json")
preferences_df.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [ ]:
preferences_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [ ]:
preferences_df.select(
    "patient_id",
    "contact.phone"
).show(truncate=False)

+----------+----------+
|patient_id|phone     |
+----------+----------+
|101       |9876500011|
|102       |NULL      |
|103       |9876500013|
|104       |9876500014|
+----------+----------+



In [ ]:
preferences_df.select(
    "patient_id",
    "contact.email"
).show(truncate=False)

+----------+---------------+
|patient_id|email          |
+----------+---------------+
|101       |rahul@gmail.com|
|102       |priya@gmail.com|
|103       |NULL           |
|104       |sneha@gmail.com|
+----------+---------------+



In [ ]:
preferences_df = spark.read.option(
    "multiline",
    "true"
).json("patient_preferences.json")
preferences_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)
preferences_flat.show()

+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       101|            Apollo|9876500011|rahul@gmail.com|
|       102|           Yashoda|      NULL|priya@gmail.com|
|       103|              Care|9876500013|           NULL|
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [ ]:
preferences_flat.filter(
    col("email").isNull()
).show()

+----------+------------------+----------+-----+
|patient_id|preferred_hospital|     phone|email|
+----------+------------------+----------+-----+
|       103|              Care|9876500013| NULL|
+----------+------------------+----------+-----+



In [ ]:
preferences_flat.filter(
    col("preferred_hospital").isNull()
).show()

+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [ ]:
preferences_flat = preferences_flat.fillna({
    "phone": "Not Available"
})
preferences_flat.show()

+----------+------------------+-------------+---------------+
|patient_id|preferred_hospital|        phone|          email|
+----------+------------------+-------------+---------------+
|       101|            Apollo|   9876500011|rahul@gmail.com|
|       102|           Yashoda|Not Available|priya@gmail.com|
|       103|              Care|   9876500013|           NULL|
|       104|              NULL|   9876500014|sneha@gmail.com|
+----------+------------------+-------------+---------------+



In [ ]:
preferences_flat = preferences_flat.fillna({
    "email": "Not Available"
})
preferences_flat.show()

+----------+------------------+-------------+---------------+
|patient_id|preferred_hospital|        phone|          email|
+----------+------------------+-------------+---------------+
|       101|            Apollo|   9876500011|rahul@gmail.com|
|       102|           Yashoda|Not Available|priya@gmail.com|
|       103|              Care|   9876500013|  Not Available|
|       104|              NULL|   9876500014|sneha@gmail.com|
+----------+------------------+-------------+---------------+



In [ ]:
patient_preferences = patients_df.join(
    preferences_flat,
    on="patient_id",
    how="inner"
)
patient_preferences.show(truncate=False)

+----------+------------+---------+---+------+-----------+----------------+------------------+-------------+---------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|preferred_hospital|phone        |email          |
+----------+------------+---------+---+------+-----------+----------------+------------------+-------------+---------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |Apollo            |9876500011   |rahul@gmail.com|
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |Yashoda           |Not Available|priya@gmail.com|
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |Care              |9876500013   |Not Available  |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |NULL              |9876500014   |sneha@gmail.com|
+----------+------------+---------+---+------+-----------+----------------+------------------+-------------+---------------+


Spark SQL

In [ ]:
patients_df.createOrReplaceTempView("patients")

In [ ]:
appointments_df.createOrReplaceTempView("appointments")

In [ ]:
spark.sql("""
SELECT *
FROM patients
""").show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [ ]:
spark.sql("""
SELECT *
FROM patients
WHERE city = 'Hyderabad'
""").show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
+----------+------------+---------+---+------+-------------+----------------+



In [ ]:
spark.sql("""
SELECT
    city,
    COUNT(*) AS patient_count
FROM patients
GROUP BY city
ORDER BY patient_count DESC
""").show()

+---------+-------------+
|     city|patient_count|
+---------+-------------+
|Bangalore|            2|
|Hyderabad|            2|
|    Kochi|            1|
|  Chennai|            1|
|   Mumbai|            1|
|  Unknown|            1|
|     Pune|            1|
|    Delhi|            1|
+---------+-------------+



In [ ]:
spark.sql("""
SELECT
    department,
    COUNT(*) AS appointment_count
FROM appointments
GROUP BY department
ORDER BY appointment_count DESC
""").show()

+-----------+-----------------+
| department|appointment_count|
+-----------+-----------------+
|Dermatology|                3|
| Cardiology|                3|
|  Neurology|                2|
|Orthopedics|                2|
+-----------+-----------------+



In [ ]:
spark.sql("""
SELECT
    department,
    ROUND(AVG(fee), 2) AS avg_fee
FROM appointments
GROUP BY department
ORDER BY avg_fee DESC
""").show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|Orthopedics| 2500.0|
|  Neurology| 2000.0|
| Cardiology| 1500.0|
|Dermatology| 666.67|
+-----------+-------+



In [ ]:
spark.sql("""
SELECT MAX(fee) AS highest_fee
FROM appointments
""").show()

+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [ ]:
spark.sql("""
SELECT
    patient_id,
    COUNT(*) AS appointment_count
FROM appointments
GROUP BY patient_id
ORDER BY appointment_count DESC
""").show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       101|                2|
|       108|                1|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [ ]:
spark.sql("""
SELECT
    p.patient_id,
    p.patient_name,
    SUM(a.fee) AS total_spent
FROM patients p
JOIN appointments a
ON p.patient_id = a.patient_id
GROUP BY p.patient_id, p.patient_name
ORDER BY total_spent DESC
LIMIT 5
""").show()

+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       110| Nisha Reddy|       2500|
|       101|Rahul Sharma|       2500|
|       104| Sneha Patel|       2500|
|       107| Arjun Verma|       2000|
|       102| Priya Reddy|       2000|
+----------+------------+-----------+



In [ ]:
ETL

In [ ]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)
appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [ ]:
preferences_df = spark.read.option(
    "multiline",
    "true"
).json("patient_preferences.json")

In [ ]:
patients_df = patients_df.fillna({
    "city": "Unknown",
    "blood_group": "Not Available"
})
preferences_flat = preferences_flat.fillna({
    "phone": "Not Available",
    "email": "Not Available",
    "preferred_hospital": "Not Specified"
})
appointments_df = appointments_df.fillna({
    "fee": 0
})

In [ ]:
hospital_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="left"
).join(
    preferences_flat,
    on="patient_id",
    how="left"
)
hospital_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----+---------+------------------+-------------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|   appointment_date| fee|  consult|preferred_hospital|        phone|          email|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+-------------------+----+---------+------------------+-------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|2025-01-15 00:00:00|1000|Completed|            Apollo|   9876500011|rahul@gmail.com|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|1500|Completed|            Apollo|   9876500011|rahul@gmail.

In [ ]:
hospital_df = hospital_df.withColumn(
    "age_group",
    when(col("age") < 18, "Child")
    .when(col("age") < 40, "Adult")
    .otherwise("Senior")
)

In [ ]:
hospital_df = hospital_df.withColumn(
    "revenue",
    coalesce(col("fee"), lit(0))
)

In [ ]:
patient_spending = hospital_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    round(sum("revenue"), 2).alias("total_spending")
)
patient_spending.show()

+----------+------------+--------------+
|patient_id|patient_name|total_spending|
+----------+------------+--------------+
|       107| Arjun Verma|          2000|
|       108|  Meera Nair|             0|
|       109|   Kiran Rao|             0|
|       110| Nisha Reddy|          2500|
|       105|  Farhan Ali|          1000|
|       101|Rahul Sharma|          2500|
|       104| Sneha Patel|          2500|
|       103|  Amit Kumar|          1500|
|       106|  Neha Singh|             0|
|       102| Priya Reddy|          2000|
+----------+------------+--------------+



In [ ]:
department_revenue = hospital_df.groupBy(
    "department"
).agg(
    round(sum("revenue"), 2).alias("department_revenue")
).orderBy(
    desc("department_revenue")
)
department_revenue.show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|Orthopedics|              5000|
|  Neurology|              4000|
| Cardiology|              3000|
|Dermatology|              2000|
|       NULL|                 0|
+-----------+------------------+



In [ ]:
hospital_df.write.mode(
    "overwrite"
).parquet(
    "hospital_analytics_parquet"
)

In [ ]:
total_patients = patients_df.count()

total_appointments = appointments_df.count()

total_revenue = hospital_df.agg(
    sum("revenue")
).collect()[0][0]

highest_spender = patient_spending.orderBy(
    desc("total_spending")
).first()

top_department = department_revenue.first()

print("=" * 50)
print("HOSPITAL ANALYTICS REPORT")
print("=" * 50)

print(f"Total Patients           : {total_patients}")
print(f"Total Appointments       : {total_appointments}")
print(f"Total Revenue            : {total_revenue}")

print("\nHighest Spending Patient")
print(f"Patient Name             : {highest_spender['patient_name']}")
print(f"Total Spending           : {highest_spender['total_spending']}")

print("\nTop Revenue Department")
print(f"Department               : {top_department['department']}")
print(f"Revenue                  : {top_department['department_revenue']}")

print("=" * 50)

HOSPITAL ANALYTICS REPORT
Total Patients           : 10
Total Appointments       : 10
Total Revenue            : 14000

Highest Spending Patient
Patient Name             : Nisha Reddy
Total Spending           : 2500

Top Revenue Department
Department               : Orthopedics
Revenue                  : 5000
